# 08. What makes an analyst call it pressure? (determinants of the P label)

Question: PFF's written guideline says pressure = opponent within 2.74 m (P) / 4.57 m (A), **closing** the space,
radius **smaller from behind** and larger at high intensity. Do analysts actually judge that way, and do
**other factors** (surrounding defenders, passer situation, pitch location) matter?

Design
- Passes labelled P vs N (open play, 52,852). Target: P.
- Factors (nearest defender = defender closest to the passer at the pass frame, raw positions):
  1. distance of the nearest defender
  2. its speed and whether it is closing (smoothed tracking velocities; PFF-stored speed as robustness)
  3. its direction: in front of (goal side) or behind the passer
  4. surrounding defenders: 2nd-nearest distance, defenders within 3 m and 5 m
  5. passer situation: passer speed, pitch location (x, |y|)
- Nested models (logistic regression with splines, game-grouped CV): how much does each block add?
- Effect shapes: predicted P-probability curves; LightGBM + SHAP for importance and interactions.
- Robustness: nearest defender visible on the broadcast only.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, SplineTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, log_loss

from football.common import KEY, cv_predict, boot_diff, GOAL_X

OUT = os.path.join("..", "outputs")
FIG = os.path.join(OUT, "figs")
E = pd.read_csv(os.path.join(OUT, "pressure_events.csv.gz"))
pl = pd.read_csv(os.path.join(OUT, "players.csv.gz"))
ev = pd.read_csv(os.path.join(OUT, "events.csv.gz"), low_memory=False)
E = E[E.label.isin(["P", "N"])].copy()
print(f"P vs N passes: {len(E):,}  (P share {E.label.eq('P').mean():.3f})")

P vs N passes: 52,852  (P share 0.423)


## 1. Factors per pass
Nearest defender from raw positions (as in notebook 05). Speeds come from the smoothed tracking (0.2 s differences),
available for all 64 games; all passes are kept, with a flag (`vel_ok`) where velocities are not trusted (speed set to 0). PFF's stored speed exists for
only 24 of the 64 games and is used as a robustness check.

In [2]:
act = ev.set_index(KEY).actor_id
P_ = pl[pl.set_index(KEY).index.isin(E.set_index(KEY).index)].copy()
P_["actor"] = act.reindex(P_.set_index(KEY).index).to_numpy()
car = P_[P_.player_id == P_.actor][KEY + ["x", "y", "speed", "visible"]].rename(
    columns={"x": "cx_", "y": "cy_", "speed": "carrier_pff_speed", "visible": "carrier_visible"})
D = P_[~P_.atk].merge(car, on=KEY)
D["dist"] = np.hypot(D.x - D.cx_, D.y - D.cy_)
D = D.sort_values(KEY + ["dist"])
near = D.groupby(KEY).head(1).set_index(KEY)
F = E.set_index(KEY)[["label", "d1", "d2", "n3", "n5", "closing1", "cos1", "speed1", "carrier_speed", "cx", "cy",
                       "vel_ok_pass", "non_event"]].copy()
F["y"] = (F.pop("label") == "P").astype(int)
F["def_speed"] = F.pop("speed1")                         # smoothed-tracking speed, all 64 games
F["def_pff_speed"] = near.speed.reindex(F.index)          # PFF-stored speed: only 24 of 64 games
F["def_visible"] = near.visible.reindex(F.index).astype(bool)
F["behind"] = (F.cos1 < 0).astype(int)
F["closing"] = (F.closing1 > 0.5).astype(int)
F["abs_y"] = F.cy.abs()
F["game_id"] = F.index.get_level_values("game_id")
n_all = len(F)
F["vel_ok"] = F.vel_ok_pass.astype(int)                     # all passes kept; speed = 0 where velocities are not trusted, flagged
print(f"P vs N passes: {n_all:,};  with trusted velocities: {F.vel_ok.sum():,};  nearest defender visible: {F.def_visible.mean():.1%};  "
      f"PFF speed available: {F.def_pff_speed.notna().mean():.1%} ({F.loc[F.def_pff_speed.notna(), 'game_id'].nunique()} games)")
F.describe().T[["mean", "50%"]].round(2)

P vs N passes: 52,852;  with trusted velocities: 41,913;  nearest defender visible: 87.0%;  PFF speed available: 38.9% (24 games)


,mean,50%
d1,5.59,4.24
d2,10.45,9.43
n3,0.41,0.00
n5,0.77,1.00
closing1,0.65,0.59
cos1,0.58,0.84
carrier_speed,1.85,1.62
cx,-3.82,-3.70
cy,0.15,0.16
y,0.42,0.00


## 2. Nested models: how much does each block of factors add?
Logistic regression; distance and continuous factors enter as cubic splines (5 knots), so non-linear shapes are allowed.
Game-grouped 5-fold CV. Gains vs the previous model: paired cluster bootstrap by game (300 resamples).

In [3]:
SPL = lambda: SplineTransformer(n_knots=5, degree=3)
BLOCKS = {
    "M1 distance": ["d1"],
    "M2 + defender speed": ["d1", "def_speed", "vel_ok"],
    "M3 + closing & direction": ["d1", "def_speed", "vel_ok", "closing", "behind"],
    "M4 + surrounding defenders": ["d1", "def_speed", "vel_ok", "closing", "behind", "d2", "n3", "n5"],
    "M5 + passer situation": ["d1", "def_speed", "vel_ok", "closing", "behind", "d2", "n3", "n5", "carrier_speed", "cx", "abs_y"],
}
CONT = {"d1", "def_speed", "d2", "carrier_speed", "cx", "abs_y"}


def model_for(cols):
    ct = ColumnTransformer([(c, SPL(), [c]) if c in CONT else (c, "passthrough", [c]) for c in cols])
    return lambda s: make_pipeline(ct, StandardScaler(), LogisticRegression(max_iter=3000, C=1.0))


def fit_blocks(Fs):
    y, g = Fs.y, Fs.game_id
    preds, rows, prev = {}, [], None
    base_ll = log_loss(y, np.full(len(y), y.mean()))
    for name, cols in BLOCKS.items():
        p = cv_predict(model_for(cols), Fs[cols].astype(float), y, g, seeds=(0,))
        preds[name] = p
        row = dict(model=name, roc_auc=roc_auc_score(y, p), log_loss=log_loss(y, p),
                   mcfadden_r2=1 - log_loss(y, p) / base_ll)
        if prev is not None:
            d, ci, _ = boot_diff(y, p, preds[prev], roc_auc_score, n=300, groups=g)
            row.update(auc_gain=d, auc_gain_ci=f"[{ci[0]:.4f}, {ci[1]:.4f}]")
        rows.append(row); prev = name
    return pd.DataFrame(rows).round(4), preds


nested, preds = fit_blocks(F)
nested

,model,roc_auc,log_loss,mcfadden_r2,auc_gain,auc_gain_ci
0,M1 distance,0.8861,0.4128,0.3939,NaN,NaN
1,M2 + defender speed,0.8884,0.4104,0.3975,0.0024,"[0.0010, 0.0036]"
2,M3 + closing & direction,0.8894,0.4096,0.3987,0.0010,"[0.0006, 0.0014]"
3,M4 + surrounding defenders,0.8902,0.4080,0.4010,0.0008,"[0.0004, 0.0011]"
4,M5 + passer situation,0.8934,0.4040,0.4069,0.0032,"[0.0025, 0.0038]"


In [4]:
# robustness: nearest defender visible on the broadcast (its position is observed, not estimated)
nested_vis, _ = fit_blocks(F[F.def_visible])
nested_vis

,model,roc_auc,log_loss,mcfadden_r2,auc_gain,auc_gain_ci
0,M1 distance,0.9077,0.3747,0.4507,NaN,NaN
1,M2 + defender speed,0.9095,0.3725,0.4540,0.0018,"[0.0008, 0.0026]"
2,M3 + closing & direction,0.9099,0.3715,0.4554,0.0005,"[0.0001, 0.0008]"
3,M4 + surrounding defenders,0.9103,0.3701,0.4575,0.0004,"[0.0001, 0.0006]"
4,M5 + passer situation,0.9112,0.3687,0.4596,0.0010,"[0.0006, 0.0013]"


In [5]:
# robustness: PFF-stored defender speed instead of tracking speed (24 games only)
Fp = F[F.def_pff_speed.notna()].copy()
Fp["def_speed"] = Fp.def_pff_speed
print(f"passes with PFF speed: {len(Fp):,} in {Fp.game_id.nunique()} games")
nested_pff, _ = fit_blocks(Fp)
nested_pff

passes with PFF speed: 20,546 in 24 games


,model,roc_auc,log_loss,mcfadden_r2,auc_gain,auc_gain_ci
0,M1 distance,0.9035,0.3872,0.4351,NaN,NaN
1,M2 + defender speed,0.9079,0.3803,0.4451,0.0044,"[0.0020, 0.0070]"
2,M3 + closing & direction,0.9082,0.3800,0.4456,0.0003,"[-0.0001, 0.0008]"
3,M4 + surrounding defenders,0.9081,0.3794,0.4464,-0.0001,"[-0.0004, 0.0001]"
4,M5 + passer situation,0.9100,0.3760,0.4515,0.0019,"[0.0012, 0.0027]"


## 3. Effect shapes — predicted probability of 'pressured'
From model M5 fitted on all passes; other factors held at their median (binary factors at 0 unless varied).

In [6]:
cols = BLOCKS["M5 + passer situation"]
m5 = model_for(cols)(0).fit(F[cols].astype(float), F.y)
med = F[cols].median()
grid = np.linspace(0.3, 8, 78)


def curve(**over):
    X = pd.DataFrame([med] * len(grid)); X["d1"] = grid
    for k, v in over.items():
        X[k] = v
    return m5.predict_proba(X[cols].astype(float))[:, 1]


fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for s, c in [(0.5, "#9ecae1"), (2.0, "#4292c6"), (4.5, "#08306b")]:
    axes[0].plot(grid, curve(def_speed=s), color=c, label=f"defender speed {s} m/s")
axes[0].set_title("Defender speed")
for (cl, b), (lab, c) in zip([(0, 0), (1, 0), (0, 1)], [("static-ish, in front", "#4292c6"), ("closing, in front", "#08306b"), ("static-ish, behind", "#e6550d")]):
    axes[1].plot(grid, curve(closing=cl, behind=b), color=c, label=lab)
axes[1].set_title("Closing and direction")
for n5, d2, c in [(1, 7.0, "#9ecae1"), (2, 4.0, "#4292c6"), (3, 3.0, "#08306b")]:
    axes[2].plot(grid, curve(n5=n5, d2=d2), color=c, label=f"{n5} defender(s) within 5 m (2nd nearest {d2:.0f} m)")
axes[2].set_title("Surrounding defenders")
for ax in axes:
    ax.axvline(2.74, ls=":", c="grey", lw=1); ax.axvline(4.57, ls=":", c="grey", lw=1)
    ax.set_xlabel("nearest defender distance (m)"); ax.legend(frameon=False, fontsize=7)
axes[0].set_ylabel("P(labelled 'pressured')")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig_determinants_curves.png"), dpi=200)

In [7]:
# distance at which P(pressured) = 0.5, by defender speed / direction / closing (the 'effective radius')
def radius50(**over):
    p = curve(**over)
    return float(grid[np.argmin(np.abs(p - 0.5))]) if (p.max() > 0.5 > p.min()) else np.nan


eff = pd.DataFrame([
    dict(situation="defender speed 0.5 m/s", r50=radius50(def_speed=0.5)),
    dict(situation="defender speed 2 m/s", r50=radius50(def_speed=2.0)),
    dict(situation="defender speed 4.5 m/s", r50=radius50(def_speed=4.5)),
    dict(situation="in front, not closing", r50=radius50(closing=0, behind=0)),
    dict(situation="in front, closing", r50=radius50(closing=1, behind=0)),
    dict(situation="behind, not closing", r50=radius50(closing=0, behind=1)),
    dict(situation="behind, closing", r50=radius50(closing=1, behind=1)),
]).round(2)
eff

,situation,r50
0,defender speed 0.5 m/s,2.8
1,defender speed 2 m/s,2.8
2,defender speed 4.5 m/s,3.1
3,"in front, not closing",2.7
4,"in front, closing",2.8
5,"behind, not closing",3.0
6,"behind, closing",3.2


## 4. Importance and interactions (LightGBM + SHAP)
Same factors, game-grouped CV for accuracy; SHAP (LightGBM `pred_contrib`) on a full fit for importance.

In [8]:
GB = dict(n_estimators=400, learning_rate=0.05, num_leaves=15, min_child_samples=50, subsample=0.8,
          subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, verbose=-1)
p_gbm = cv_predict(lambda s: lgb.LGBMClassifier(random_state=s, **GB), F[cols].astype(float), F.y, F.game_id, seeds=(0,))
print(f"LightGBM (M5 factors) CV ROC-AUC = {roc_auc_score(F.y, p_gbm):.4f}   vs logistic M5 = {roc_auc_score(F.y, preds['M5 + passer situation']):.4f}")
gbm = lgb.LGBMClassifier(random_state=0, **GB).fit(F[cols].astype(float), F.y)
sh = gbm.predict(F[cols].astype(float), pred_contrib=True)[:, :-1]
imp = pd.DataFrame({"factor": cols, "mean_abs_shap": np.abs(sh).mean(0)}).sort_values("mean_abs_shap", ascending=False)
imp["share"] = imp.mean_abs_shap / imp.mean_abs_shap.sum()
fig, ax = plt.subplots(figsize=(5, 3.4))
ax.barh(imp.factor[::-1], imp.share[::-1], color="#4292c6")
ax.set_xlabel("share of mean |SHAP|"); ax.set_title("What drives the analysts' 'pressured' label")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig_determinants_shap.png"), dpi=200)
imp.round(3)

LightGBM (M5 factors) CV ROC-AUC = 0.9061   vs logistic M5 = 0.8934


,factor,mean_abs_shap,share
0,d1,1.532,0.572
10,abs_y,0.184,0.069
6,n3,0.159,0.059
5,d2,0.134,0.050
9,cx,0.125,0.046
1,def_speed,0.116,0.043
7,n5,0.112,0.042
4,behind,0.099,0.037
3,closing,0.097,0.036
8,carrier_speed,0.072,0.027


## 5. Guideline vs practice (summary table for the paper)

In [9]:
rule = pd.read_csv(os.path.join(OUT, "pressure_rule_agreement.csv"))
summary = pd.DataFrame([
    ["Distance threshold", "P within 2.74 m (A within 4.57 m)",
     f"50% line at {radius50():.2f} m (median speed, in front, not closing)"],
    ["Closing required", "static / moving away = no pressure",
     f"in front: 50% line {radius50(closing=0, behind=0):.2f} m not closing vs {radius50(closing=1, behind=0):.2f} m closing"],
    ["Intensity widens radius", "e.g. 3.66 m at full sprint",
     f"50% line {radius50(def_speed=0.5):.2f} m (0.5 m/s) → {radius50(def_speed=4.5):.2f} m (4.5 m/s)"],
    ["Smaller radius from behind", "reduced radius behind the carrier",
     f"50% line in front {radius50(closing=0, behind=0):.2f} m vs behind {radius50(closing=0, behind=1):.2f} m"],
    ["Written rule on tracking", "—", f"weighted κ {rule.weighted_kappa.iloc[-1]:.2f} vs distance-only {rule.weighted_kappa.iloc[0]:.2f}"],
], columns=["guideline element", "written guideline", "analysts in practice (this data)"])
summary

,guideline element,written guideline,analysts in practice (this data)
0,Distance threshold,P within 2.74 m (A within 4.57 m),"50% line at 2.80 m (median speed, in front, no..."
1,Closing required,static / moving away = no pressure,in front: 50% line 2.70 m not closing vs 2.80 ...
2,Intensity widens radius,e.g. 3.66 m at full sprint,50% line 2.80 m (0.5 m/s) → 3.10 m (4.5 m/s)
3,Smaller radius from behind,reduced radius behind the carrier,50% line in front 2.70 m vs behind 3.00 m
4,Written rule on tracking,—,weighted κ 0.43 vs distance-only 0.59


## Save

In [10]:
nested.to_csv(os.path.join(OUT, "determinants_nested.csv"), index=False)
nested_vis.to_csv(os.path.join(OUT, "determinants_nested_visible.csv"), index=False)
nested_pff.to_csv(os.path.join(OUT, "determinants_nested_pffspeed.csv"), index=False)
eff.to_csv(os.path.join(OUT, "determinants_effective_radius.csv"), index=False)
imp.to_csv(os.path.join(OUT, "determinants_shap.csv"), index=False)
summary.to_csv(os.path.join(OUT, "determinants_guideline_vs_practice.csv"), index=False)
print("saved")

saved
